In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [13]:
df = pd.read_csv('data.csv')
df.head(3)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [14]:
for column in df.columns:
    print(column, df[column].unique())

customerID ['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']
gender ['Female' 'Male']
SeniorCitizen [0 1]
Partner ['Yes' 'No']
Dependents ['No' 'Yes']
tenure [ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26  0
 39]
PhoneService ['No' 'Yes']
MultipleLines ['No phone service' 'No' 'Yes']
InternetService ['DSL' 'Fiber optic' 'No']
OnlineSecurity ['No' 'Yes' 'No internet service']
OnlineBackup ['Yes' 'No' 'No internet service']
DeviceProtection ['No' 'Yes' 'No internet service']
TechSupport ['No' 'Yes' 'No internet service']
StreamingTV ['No' 'Yes' 'No internet service']
StreamingMovies ['No' 'Yes' 'No internet service']
Contract ['Month-to-month' 'One year' 'Two year']
PaperlessBilling ['Yes' 'No']
PaymentMethod ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (a

In [15]:
df.drop(['customerID'], axis=1, inplace=True)
dummies_contract = pd.get_dummies(df['Contract'])
dummies_InternetService = pd.get_dummies(df['InternetService'])
dummies_payment = pd.get_dummies(df['PaymentMethod'])
df = pd.concat([df, dummies_contract, dummies_payment, dummies_InternetService], axis=1)
df.drop(['Contract', 'PaymentMethod', 'InternetService', 'No'], axis=1, inplace=True)
df.head(3)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,Churn,Month-to-month,One year,Two year,Bank transfer (automatic),Credit card (automatic),Electronic check,Mailed check,DSL,Fiber optic
0,Female,0,Yes,No,1,No,No phone service,No,Yes,No,...,No,1,0,0,0,0,1,0,1,0
1,Male,0,No,No,34,Yes,No,Yes,No,Yes,...,No,0,1,0,0,0,0,1,1,0
2,Male,0,No,No,2,Yes,No,Yes,Yes,No,...,Yes,1,0,0,0,0,0,1,1,0


In [16]:
df.isna().any()

gender                       False
SeniorCitizen                False
Partner                      False
Dependents                   False
tenure                       False
PhoneService                 False
MultipleLines                False
OnlineSecurity               False
OnlineBackup                 False
DeviceProtection             False
TechSupport                  False
StreamingTV                  False
StreamingMovies              False
PaperlessBilling             False
MonthlyCharges               False
TotalCharges                 False
Churn                        False
Month-to-month               False
One year                     False
Two year                     False
Bank transfer (automatic)    False
Credit card (automatic)      False
Electronic check             False
Mailed check                 False
DSL                          False
Fiber optic                  False
dtype: bool

In [17]:
df['gender'] = df['gender'].apply(lambda x: 0 if x == 'Female' else 1)

cols_to_convert = ['Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'Churn']

df[cols_to_convert] = df[cols_to_convert].applymap(lambda x: 1 if x == 'Yes' else 0)
df = df[df['TotalCharges'] != ' ']
df['TotalCharges'] = df['TotalCharges'].astype(float)
df.dtypes


gender                         int64
SeniorCitizen                  int64
Partner                        int64
Dependents                     int64
tenure                         int64
PhoneService                   int64
MultipleLines                  int64
OnlineSecurity                 int64
OnlineBackup                   int64
DeviceProtection               int64
TechSupport                    int64
StreamingTV                    int64
StreamingMovies                int64
PaperlessBilling               int64
MonthlyCharges               float64
TotalCharges                 float64
Churn                          int64
Month-to-month                 uint8
One year                       uint8
Two year                       uint8
Bank transfer (automatic)      uint8
Credit card (automatic)        uint8
Electronic check               uint8
Mailed check                   uint8
DSL                            uint8
Fiber optic                    uint8
dtype: object

In [18]:
import mlflow
from sklearn.metrics import mean_squared_error, r2_score, f1_score, mean_absolute_error
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(df.drop(['Churn'], axis=1), df['Churn'], test_size=0.2, random_state=3)
(X_train['TotalCharges']==' ').any()
# df['TotalCharges']

False

In [19]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("CustomerChurnPrediction")

<Experiment: artifact_location='file:///c:/Saqlain/projects_python/MLOps/customer_churn_prediction/mlruns/1', creation_time=1700246056359, experiment_id='1', last_update_time=1700246056359, lifecycle_stage='active', name='CustomerChurnPrediction', tags={}>

## Logistic Regression

In [20]:
mlflow.sklearn.autolog(disable=True)

with mlflow.start_run(run_name="logistic_reg"):
    mlflow.set_tag("model_name", "LogisticRegression")
    lr = LogisticRegression()
    lr.fit(X_train, y_train)
    lr_predict = lr.predict(X_test)

    metrics = {

        "test_mse" : mean_squared_error(y_test, lr_predict, squared=False),
        "test_r2" : r2_score(y_test, lr_predict),
        "test_f1" : f1_score(y_test, lr_predict),
        "test_mae" : mean_absolute_error(y_test, lr_predict)
    }

    mlflow.log_metrics(metrics=metrics)
    mlflow.sklearn.log_model(lr, "sk_models")


## Decision Tree Classifier

In [21]:
mlflow.sklearn.autolog(disable=True)


params = {
    'random_state' : [x for x in range(40, 62)]
}
    
for i in range(0, len(params['random_state'])):
    with mlflow.start_run(run_name=f"decision_tree{i}"):
        mlflow.set_tag("model_name", f"DecisionTreeClassifier{i}")
        mlflow.log_params({'random_state' : params['random_state'][i]})
        dc = DecisionTreeClassifier(random_state=params['random_state'][i])
        dc.fit(X_train, y_train)
        dc_predict = dc.predict(X_test)

        metrics = {

            "test_mse" : mean_squared_error(y_test, dc_predict, squared=False),
            "test_r2" : r2_score(y_test, dc_predict),
            "test_f1" : f1_score(y_test, dc_predict),
            "test_mae" : mean_absolute_error(y_test, dc_predict)
        }

        mlflow.log_metrics(metrics=metrics)
        mlflow.sklearn.log_model(dc, "sk_models")

In [22]:
mlflow.sklearn.autolog(disable=True)

params = {
    'n_estimators' : 20
}

with mlflow.start_run(run_name="xgbc"):
    mlflow.set_tag("model_name", "XGBClassifier")
    mlflow.log_params(params)
    xgbc = XGBClassifier(n_estimators = params['n_estimators'], enable_categorical=True)
    xgbc.fit(X_train, y_train)
    xgbc_predict = xgbc.predict(X_test)
    metrics = {

            "test_mse" : mean_squared_error(y_test, xgbc_predict, squared=False),
            "test_r2" : r2_score(y_test, xgbc_predict),
            "test_f1" : f1_score(y_test, xgbc_predict),
            "test_mae" : mean_absolute_error(y_test, xgbc_predict)
        }
    mlflow.log_metrics(metrics)
    mlflow.xgboost.log_model(xgbc, "xgb_models")


c:\Users\LENOVO\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\core.py:160: UserWarning: [12:48:57] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0cec3277c4d9d0165-1\xgboost\xgboost-ci-windows\src\c_api\c_api.cc:1240: Saving into deprecated binary model format, please consider using `json` or `ubj`. Model format will default to JSON in XGBoost 2.2 if not specified.
  warnings.warn(smsg, UserWarning)
